# 15 — Advanced Multi-Tool / Multi-Server Workflows

This capstone notebook combines everything from notebooks 01–14 into more advanced patterns: connecting to multiple MCP servers at once, and orchestrating multi-step Azure workflows.

## Multi-server MCP clients

A single host can hold *multiple* MCP clients simultaneously — for example, the Azure MCP Server for cloud operations plus a separate MCP server for local filesystem access, a ticketing system, or the [Foundry MCP Server](https://learn.microsoft.com/en-us/azure/developer/azure-mcp-server/how-to/deploy-remote-mcp-server-microsoft-foundry). Each client's tools get merged into one combined `tools=` list passed to the LLM; the client that actually executes a given tool call is chosen by matching the tool name back to its owning session.

In [ ]:
# Conceptual sketch: dispatch a tool call to whichever connected client
# actually owns that tool name. (Only the Azure MCP Server is live here;
# a second server would be added the same way.)
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.mcp_client import AzureMcpClient, build_server_params

class MultiServerRouter:
    def __init__(self):
        self.clients_by_tool: dict[str, AzureMcpClient] = {}

    async def register(self, client: AzureMcpClient):
        for tool in await client.list_tools():
            self.clients_by_tool[tool.name] = client

    async def call_tool(self, name: str, arguments: dict):
        client = self.clients_by_tool[name]
        return await client.call_tool(name, arguments)

router = MultiServerRouter()
azure_client = AzureMcpClient(build_server_params(read_only=True))
await azure_client.__aenter__()
await router.register(azure_client)
print(f"Router knows about {len(router.clients_by_tool)} tools")

## The Azure Skills Plugin

For production-grade multi-step workflows, Microsoft ships the [Azure Skills Plugin](https://github.com/microsoft/azure-skills): 26+ reusable, version-controlled skills (`azure-prepare`, `azure-validate`, `azure-deploy`, `azure-diagnostics`, `azure-cost`, ...) that add structured guardrails on top of the raw tools you've been calling directly in this course.